# 04. Pelatihan, Efisiensi Komputasi, dan Evaluasi 5 Model Klasifikasi

Notebook ini menjalankan **Tahap 4 metodologi penelitian** — melatih dan mengevaluasi secara penuh **5 algoritma pembelajaran mesin** (*Support Vector Machine*, *Logistic Regression*, *Naïve Bayes*, *Random Forest*, dan *Gradient Boosting*) dari sumber tunggal `data/basic_articles_dataset.csv` untuk kedua pendekatan:
1. **Dengan Semantik *Frame*** (`ngram_range=(2, 3)`, retensi *stopwords*, split `85% / 10% / 5%`).
2. **Medan Makna / Tanpa Semantik *Frame*** (`ngram_range=(1, 1)`, penghapusan *stopwords*, split `80% / 15% / 5%`).

Seluruh laporan metrik (`results/metrics/`), log CPU/RAM (`results/metrics/cpu_and_memory/`), model terbaik SVM (`models/svm_frame_semantik.joblib` dan `models/svm_medan_makna.joblib`), serta grafik perbandingan dan *Confusion Matrix* dihasilkan langsung dari proses pelatihan di notebook ini.


In [ ]:
import sys
from pathlib import Path

# Resolve root repositori secara relatif
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "src" / "arabic_news").exists() else cwd.parent
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Repository root: {REPO_ROOT.name}")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from arabic_news.config import (
    FIGURES_DIR,
    LABELS,
    LABELS_ID,
    METRICS_DIR,
    MODELS_DIR,
    RANDOM_SEED,
    resolve_data_file,
)
from arabic_news.evaluation import (
    evaluate_model,
    prepare_medan_and_frame_splits,
    save_evaluation_metrics_txt,
    save_resource_log_txt,
    train_classifier_with_monitoring,
)
from arabic_news.features import create_tfidf_vectorizer
from arabic_news.models import build_deployable_svm_bundle, save_deployable_svm_bundle
from arabic_news.preprocessing import reshape_arabic_for_plot

plt.rcParams["figure.dpi"] = 120


## 4.1 Persiapan Split & Vektorisasi TF-IDF Kedua Pendekatan dari `basic_articles_dataset.csv`


In [ ]:
basic_csv = resolve_data_file("basic_articles_dataset.csv")
df_basic = pd.read_csv(basic_csv, low_memory=False)
splits = prepare_medan_and_frame_splits(df_basic, random_state=RANDOM_SEED)

# Jalur Semantik Frame (2,3)
tr_f_df = splits["frame_semantik"]["train_df"]
val_f_df = splits["frame_semantik"]["val_df"]
vec_frame = create_tfidf_vectorizer(ngram_range=(2, 3))
X_tr_frame = vec_frame.fit_transform(tr_f_df["text"])
X_val_frame = vec_frame.transform(val_f_df["text"])
y_tr_frame = tr_f_df["label"]
y_val_frame = val_f_df["label"]
print(f"TF-IDF Semantik Frame (2,3) : train={X_tr_frame.shape}, val={X_val_frame.shape}")

# Jalur Medan Makna (1,1)
tr_m_txt = splits["medan_makna"]["train_text"]
val_m_txt = splits["medan_makna"]["val_text"]
y_tr_medan = splits["medan_makna"]["train_labels"]
y_val_medan = splits["medan_makna"]["val_labels"]
vec_medan = create_tfidf_vectorizer(ngram_range=(1, 1))
X_tr_medan = vec_medan.fit_transform(tr_m_txt)
X_val_medan = vec_medan.transform(val_m_txt)
print(f"TF-IDF Medan Makna (1,1)    : train={X_tr_medan.shape}, val={X_val_medan.shape}")


## 4.2 Pelatihan, Pemantauan CPU/RAM, & Evaluasi Penuh 5 Algoritma untuk Kedua Pendekatan

Melatih kelima algoritma (`SVC`, `LogisticRegression`, `MultinomialNB`, `RandomForest`, `GradientBoosting`), mengukur konsumsi RAM puncak (MB), penggunaan CPU (%), waktu evaluasi (detik), *Accuracy*, *Precision*, *Recall*, *F1-Score*, serta menyimpan model terbaik `LinearSVC(C=10)` versi deployable (`<50 MB`) ke `models/`.

*(Catatan: pelatihan Random Forest & Gradient Boosting pada jutaan fitur bigram/trigram membutuhkan memori besar dan waktu eksekusi lebih lama dibanding model linear.)*


In [ ]:
MODELS_ORDER = [
    ("SVC", "SVM (LinearSVC C=10)", "svm.txt"),
    ("LogisticRegression", "Logistic Regression", "logistic.txt"),
    ("MultinomialNB", "Naïve Bayes", "naive.txt"),
    ("RandomForest", "Random Forest", "forest.txt"),
    ("GradientBoosting", "Gradient Boosting", "gradient.txt"),
]

eff_rows = []
perf_rows = []
svc_eval_by_scheme = {}

for key, display_name, res_file in MODELS_ORDER:
    print(f"Melatih {display_name} (Semantik Frame 2,3)...")
    clf_f, res_f = train_classifier_with_monitoring(
        key, X_tr_frame, y_tr_frame, mode="frame", random_state=RANDOM_SEED
    )
    ev_f = evaluate_model(clf_f, X_val_frame, y_val_frame)
    save_evaluation_metrics_txt(ev_f, key, METRICS_DIR / "frame_semantik" / f"no_lemma_evaluation_metrics_{key}.txt")
    save_resource_log_txt(res_f, key, METRICS_DIR / "cpu_and_memory" / "frame_semantik" / res_file)

    print(f"Melatih {display_name} (Medan Makna 1,1)...")
    clf_m, res_m = train_classifier_with_monitoring(
        key, X_tr_medan, y_tr_medan, mode="frame", random_state=RANDOM_SEED
    )
    ev_m = evaluate_model(clf_m, X_val_medan, y_val_medan)
    save_evaluation_metrics_txt(ev_m, key, METRICS_DIR / "medan_semantik" / f"no_lemma_evaluation_metrics_{key}.txt")
    save_resource_log_txt(res_m, key, METRICS_DIR / "cpu_and_memory" / "medan_semantik" / res_file)

    if key == "SVC":
        svc_eval_by_scheme["frame"] = ev_f
        svc_eval_by_scheme["medan"] = ev_m

        bundle_frame = build_deployable_svm_bundle(
            vec_frame, clf_f, mode="frame_semantik", ngram_range=(2, 3),
            max_features=700_000, full_accuracy=ev_f["accuracy"],
        )
        ev_f_compact = evaluate_model(
            bundle_frame["classifier"],
            bundle_frame["vectorizer"].transform(val_f_df["text"]),
            y_val_frame,
        )
        bundle_frame["compact_accuracy"] = ev_f_compact["accuracy"]
        save_deployable_svm_bundle(bundle_frame, MODELS_DIR / "svm_frame_semantik.joblib")

        bundle_medan = build_deployable_svm_bundle(
            vec_medan, clf_m, mode="medan_makna", ngram_range=(1, 1),
            max_features=500_000, full_accuracy=ev_m["accuracy"],
        )
        ev_m_compact = evaluate_model(
            bundle_medan["classifier"],
            bundle_medan["vectorizer"].transform(val_m_txt),
            y_val_medan,
        )
        bundle_medan["compact_accuracy"] = ev_m_compact["accuracy"]
        save_deployable_svm_bundle(bundle_medan, MODELS_DIR / "svm_medan_makna.joblib")
        print(
            f"Model terbaik SVM tersimpan: Frame compact acc={ev_f_compact['accuracy']:.4f} | "
            f"Medan compact acc={ev_m_compact['accuracy']:.4f}"
        )

    w_f = ev_f["report_dict"]["weighted avg"]
    w_m = ev_m["report_dict"]["weighted avg"]
    mem_inc = (
        round((res_f["max_memory_mb"] - res_m["max_memory_mb"]) / res_m["max_memory_mb"] * 100, 2)
        if res_m["max_memory_mb"]
        else 0.0
    )

    eff_rows.append({
        "Model": display_name,
        "Waktu Frame (s)": round(ev_f["execution_time_seconds"], 4),
        "Waktu Medan (s)": round(ev_m["execution_time_seconds"], 4),
        "RAM Frame (MB)": round(res_f["max_memory_mb"], 2),
        "RAM Medan (MB)": round(res_m["max_memory_mb"], 2),
        "Kenaikan RAM (%)": mem_inc,
        "CPU Frame (%)": round(res_f["avg_cpu_percent"], 2),
        "CPU Medan (%)": round(res_m["avg_cpu_percent"], 2),
    })
    perf_rows.append({
        "Model": display_name,
        "Acc Frame (2,3)": round(ev_f["accuracy"], 4),
        "Prec Frame": round(w_f["precision"], 4),
        "Rec Frame": round(w_f["recall"], 4),
        "F1 Frame": round(w_f["f1-score"], 4),
        "Support Frame": int(w_f["support"]),
        "Acc Medan (1,1)": round(ev_m["accuracy"], 4),
        "Prec Medan": round(w_m["precision"], 4),
        "Rec Medan": round(w_m["recall"], 4),
        "F1 Medan": round(w_m["f1-score"], 4),
        "Support Medan": int(w_m["support"]),
    })

eff_df = pd.DataFrame(eff_rows)
perf_df = pd.DataFrame(perf_rows)
eff_df.to_csv(METRICS_DIR / "efficiency_comparison.csv", index=False)
perf_df.to_csv(METRICS_DIR / "effectiveness_comparison.csv", index=False)

display(eff_df)
display(perf_df)


## 4.3 Performa Per-Kategori, Grafik Komparasi & *Confusion Matrix* Model Terbaik SVM


In [ ]:
ev_svc_f = svc_eval_by_scheme["frame"]
rep_dict = ev_svc_f["report_dict"]

per_class_df = pd.DataFrame([
    {
        "Label": cid,
        "Kategori": f"{LABELS[cid]} ({LABELS_ID[cid]})",
        "Precision": round(rep_dict[str(cid)]["precision"], 4),
        "Recall": round(rep_dict[str(cid)]["recall"], 4),
        "F1-Score": round(rep_dict[str(cid)]["f1-score"], 4),
        "Support": int(rep_dict[str(cid)]["support"]),
    }
    for cid in range(1, 12)
])
display(per_class_df)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
x = np.arange(len(perf_df))
w = 0.35
short_names = ["SVM", "LogReg", "NaiveBayes", "RandomForest", "GradBoost"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].bar(x - w/2, perf_df["Acc Frame (2,3)"] * 100, w, label="Dengan Frame (2,3)")
axes[0].bar(x + w/2, perf_df["Acc Medan (1,1)"] * 100, w, label="Tanpa Frame / Medan (1,1)")
axes[0].set_title("Perbandingan Akurasi (%)")
axes[0].set_ylim(75, 95)
axes[0].set_xticks(x, short_names, rotation=20)
axes[0].legend()

axes[1].bar(x - w/2, eff_df["Waktu Frame (s)"], w, label="Dengan Frame (2,3)")
axes[1].bar(x + w/2, eff_df["Waktu Medan (s)"], w, label="Tanpa Frame / Medan (1,1)")
axes[1].set_title("Waktu Evaluasi (detik)")
axes[1].set_xticks(x, short_names, rotation=20)
axes[1].legend()

axes[2].bar(x - w/2, eff_df["RAM Frame (MB)"], w, label="Dengan Frame (2,3)")
axes[2].bar(x + w/2, eff_df["RAM Medan (MB)"], w, label="Tanpa Frame / Medan (1,1)")
axes[2].set_title("Konsumsi RAM Puncak (MB)")
axes[2].set_xticks(x, short_names, rotation=20)
axes[2].legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / "model_comparison.png", dpi=150)
plt.show()

labels_ar = [reshape_arabic_for_plot(LABELS[i]) for i in range(1, 12)]
fig_cm = plt.figure(figsize=(10, 8.5))
sns.heatmap(
    ev_svc_f["confusion_matrix"],
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels_ar,
    yticklabels=labels_ar,
)
plt.title(f"Confusion Matrix SVM Dengan Semantik Frame (Akurasi: {ev_svc_f['accuracy']*100:.2f}%)")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.tight_layout()
fig_cm.savefig(FIGURES_DIR / "confusion_matrix_svm_frame.png", dpi=150)
plt.show()
